In [1]:
# ============================================================
# 09_shap_dependence.ipynb
# SHAP Dependence Plots — Top Features per Disease
#
# Shows how individual feature values relate to SHAP values,
# revealing non-linear effects and interaction patterns.
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')
BASE_FIG = os.path.join('..', 'outputs', 'figures')

DISEASE_REGISTRY = {
    'htn': {
        'target'      : 'HE_HP',
        'label'       : 'Hypertension',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
        'top_features': ['HE_wc', 'HE_wt', 'HE_BMI'],
    },
    'dm': {
        'target'      : 'HE_DM_HbA1c',
        'label'       : 'Diabetes',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
        'top_features': ['HE_wc', 'HE_wt', 'HE_BMI'],
    },
    'dys': {
        'target'      : 'HE_HCHOL',
        'label'       : 'Dyslipidemia',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
        'top_features': ['HE_wc', 'HE_wt', 'HE_BMI'],
    },
}

NON_ACTIONABLE = [
    'ID', 'year', 'age_group', 'age', 'sex',
    'edu', 'incm', 'ho_incm',
    'BE3_71', 'BE3_81', 'BP1', 'mh_stress',
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3',
]

FEATURE_LABEL = {
    'HE_wc'  : 'Waist circumference (cm)',
    'HE_wt'  : 'Body weight (kg)',
    'HE_BMI' : 'BMI (kg/m²)',
    'N_NA'   : 'Sodium intake (mg)',
    'N_K'    : 'Potassium intake (mg)',
    'N_SUGAR': 'Sugar intake (g)',
    'N_CHO'  : 'Carbohydrate intake (g)',
    'N_EN'   : 'Energy intake (kcal)',
    'N_FAT'  : 'Fat intake (g)',
    'N_CHOL' : 'Dietary cholesterol (mg)',
    'L_BR_FQ': 'Breakfast frequency',
    'BS3_1'  : 'Current smoking',
    'BD1_11' : 'Alcohol drinking frequency',
}

RANDOM_SEED = 42
print("Configuration loaded.")


# ─────────────────────────────────────────────
# Cell 2 | SHAP Dependence + Interaction Plots
# ─────────────────────────────────────────────
for d_key, d_info in DISEASE_REGISTRY.items():
    target       = d_info['target']
    label        = d_info['label']
    feat         = d_info['feature_cols']
    top_features = d_info['top_features']

    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    drop  = NON_ACTIONABLE + [target]
    fcols = [c for c in df.columns
              if c not in drop and c in feat]
    X = df[fcols].astype(float)

    model = joblib.load(
        os.path.join(BASE_MOD,
                     f"xgb_{d_key}_total.joblib")
    )

    # Sample for SHAP (faster)
    X_sample = X.sample(
        min(2000, len(X)), random_state=RANDOM_SEED
    )

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    print(f"\n  {label} — SHAP dependence plots")

    # ── Figure: 3 dependence plots side by side ──
    fig, axes = plt.subplots(
        1, len(top_features),
        figsize=(5 * len(top_features), 5)
    )
    if len(top_features) == 1:
        axes = [axes]

    for ax, feat_name in zip(axes, top_features):
        if feat_name not in fcols:
            continue
        feat_idx = fcols.index(feat_name)
        feat_lbl = FEATURE_LABEL.get(
            feat_name, feat_name
        )

        shap_vals = shap_values[:, feat_idx]
        feat_vals = X_sample[feat_name].values

        # Color by age group if available
        if 'age_group' in df.columns:
            age_vals = df.loc[
                X_sample.index, 'age_group'
            ].map({'young': 0, 'middle': 1,
                   'elderly': 2}).values
            sc = ax.scatter(
                feat_vals, shap_vals,
                c=age_vals, cmap='RdYlBu_r',
                alpha=0.4, s=10, vmin=0, vmax=2
            )
            cbar = plt.colorbar(sc, ax=ax,
                                 ticks=[0, 1, 2])
            cbar.set_ticklabels(
                ['Young', 'Middle', 'Elderly']
            )
            cbar.ax.tick_params(labelsize=8)
        else:
            ax.scatter(
                feat_vals, shap_vals,
                alpha=0.3, s=10, color='#2c7bb6'
            )

        # Trend line (LOWESS-like via polynomial)
        z = np.polyfit(feat_vals, shap_vals, 3)
        p = np.poly1d(z)
        x_line = np.linspace(
            feat_vals.min(), feat_vals.max(), 100
        )
        ax.plot(x_line, p(x_line), 'r-',
                linewidth=2, label='Trend')

        ax.axhline(0, color='grey', linestyle='--',
                   linewidth=0.8, alpha=0.6)
        ax.set_xlabel(feat_lbl, fontsize=10)
        ax.set_ylabel('SHAP value', fontsize=10)
        ax.set_title(
            f'{feat_lbl}\n(colored by age group)',
            fontsize=10, fontweight='bold'
        )
        ax.grid(linestyle='--', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle(
        f'SHAP Dependence Plots — {label}\n'
        f'(Top-3 features, N=2,000 sample)',
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    fig_path = os.path.join(
        BASE_FIG,
        f'fig_shap_dependence_{d_key}.png'
    )
    plt.savefig(fig_path, dpi=150,
                bbox_inches='tight')
    plt.close()
    print(f"    Saved : {fig_path}")

    # ── SHAP interaction: wc × wt ────────────
    fig2, ax2 = plt.subplots(figsize=(7, 5))

    if ('HE_wc' in fcols and
            'HE_wt' in fcols):
        wc_idx = fcols.index('HE_wc')
        wt_vals = X_sample['HE_wt'].values
        wc_shap = shap_values[:, wc_idx]

        sc2 = ax2.scatter(
            X_sample['HE_wc'].values,
            wc_shap,
            c=wt_vals, cmap='viridis',
            alpha=0.4, s=12
        )
        cbar2 = plt.colorbar(sc2, ax=ax2)
        cbar2.set_label('Body weight (kg)',
                        fontsize=9)
        ax2.axhline(0, color='grey',
                    linestyle='--', linewidth=0.8)
        ax2.set_xlabel('Waist circumference (cm)',
                       fontsize=11)
        ax2.set_ylabel(
            'SHAP value for waist circumference',
            fontsize=11
        )
        ax2.set_title(
            f'SHAP Interaction: '
            f'Waist Circ. × Body Weight\n'
            f'— {label}',
            fontsize=11
        )
        ax2.grid(linestyle='--', alpha=0.3)
        ax2.spines[['top', 'right']]\
           .set_visible(False)
        plt.tight_layout()

        int_path = os.path.join(
            BASE_FIG,
            f'fig_shap_interaction_{d_key}.png'
        )
        plt.savefig(int_path, dpi=150,
                    bbox_inches='tight')
        plt.close()
        print(f"    Interaction saved : {int_path}")

print("\n=== 09_shap_dependence.ipynb complete ===")

C:\Users\miy\miniconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration loaded.

  Hypertension — SHAP dependence plots
    Saved : ..\outputs\figures\fig_shap_dependence_htn.png
    Interaction saved : ..\outputs\figures\fig_shap_interaction_htn.png

  Diabetes — SHAP dependence plots
    Saved : ..\outputs\figures\fig_shap_dependence_dm.png
    Interaction saved : ..\outputs\figures\fig_shap_interaction_dm.png

  Dyslipidemia — SHAP dependence plots
    Saved : ..\outputs\figures\fig_shap_dependence_dys.png
    Interaction saved : ..\outputs\figures\fig_shap_interaction_dys.png

=== 09_shap_dependence.ipynb complete ===
